# Preprocessing: Daily CAISO Load Model

In this notebook, I prepare the daily dataset for modeling.

### Goals
- load and inspect the daily dataset
- convert dates and sort chronologically
- remove leakage-risk columns
- create cyclical calendar features
- keep categorical features in raw form for pipeline encoding later
- split the data into train, validation, and test sets using time order
- save clean, unscaled datasets for the modeling notebook

This notebook does **not** scale features or one-hot encode categorical variables. Those steps will happen later inside a scikit-learn pipeline.

In [1]:
import pandas as pd
import numpy as np

In [3]:
# Load daily modeling file
df = pd.read_csv("../data/raw/model_input_daily.csv")

# Make a copy so we don't accidentally overwrite the raw import
df = df.copy()

print(df.shape)
df.head()

(365, 52)


,datetime,load_mw_mean,load_mw_max,load_mw_min,load_mw_std,la_temp_mean,la_temp_max,la_temp_min,la_temp_std,la_rhum_mean,...,sd_wspd_max,sd_wspd_std,target_load_mw_mean,year,month,day,day_of_week,day_name,day_of_year,is_weekend
0,2025-01-01,22574.940652,24870.967744,19916.727203,1631.902702,11.875000,19.2,6.8,4.169272,68.125000,...,16.6,4.159214,22574.940652,2025,1,1,2,Wednesday,1,0
1,2025-01-02,24140.178717,27473.091131,19700.929643,2522.330349,13.683333,23.6,7.7,5.340710,56.958333,...,11.2,4.178820,24140.178717,2025,1,2,3,Thursday,2,0
2,2025-01-03,24641.971448,27373.142624,21211.978156,1933.071405,13.587500,21.4,9.4,3.157092,63.458333,...,16.6,5.111106,24641.971448,2025,1,3,4,Friday,3,0
3,2025-01-04,23242.359045,26109.311246,20251.253931,1748.794413,12.391667,17.3,8.3,2.891504,80.875000,...,16.6,5.137240,23242.359045,2025,1,4,5,Saturday,4,1
4,2025-01-05,22584.144627,25099.710522,20115.855130,1519.459334,13.295833,23.3,7.3,5.407039,50.875000,...,13.0,4.360960,22584.144627,2025,1,5,6,Sunday,5,1


## Inspect the dataset

Before changing anything, I want to check:
- what columns are present
- data types
- whether there are missing values
- whether there are duplicate rows

In [4]:
# View columns
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False).head(15))

print("\nDuplicate rows:", df.duplicated().sum())

Columns:
['datetime', 'load_mw_mean', 'load_mw_max', 'load_mw_min', 'load_mw_std', 'la_temp_mean', 'la_temp_max', 'la_temp_min', 'la_temp_std', 'la_rhum_mean', 'la_rhum_max', 'la_rhum_min', 'la_rhum_std', 'la_prcp_sum', 'la_prcp_max', 'la_wspd_mean', 'la_wspd_max', 'la_wspd_std', 'sf_temp_mean', 'sf_temp_max', 'sf_temp_min', 'sf_temp_std', 'sf_rhum_mean', 'sf_rhum_max', 'sf_rhum_min', 'sf_rhum_std', 'sf_prcp_sum', 'sf_prcp_max', 'sf_wspd_mean', 'sf_wspd_max', 'sf_wspd_std', 'sd_temp_mean', 'sd_temp_max', 'sd_temp_min', 'sd_temp_std', 'sd_rhum_mean', 'sd_rhum_max', 'sd_rhum_min', 'sd_rhum_std', 'sd_prcp_sum', 'sd_prcp_max', 'sd_wspd_mean', 'sd_wspd_max', 'sd_wspd_std', 'target_load_mw_mean', 'year', 'month', 'day', 'day_of_week', 'day_name', 'day_of_year', 'is_weekend']

Data types:
datetime                   str
load_mw_mean           float64
load_mw_max            float64
load_mw_min            float64
load_mw_std            float64
la_temp_mean           float64
la_temp_max          

In [5]:
# Convert datetime column to pandas datetime
df["datetime"] = pd.to_datetime(df["datetime"])

# Sort by date just to be safe
df = df.sort_values("datetime").reset_index(drop=True)

# Confirm
df[["datetime"]].head()

,datetime
0,2025-01-01
1,2025-01-02
2,2025-01-03
3,2025-01-04
4,2025-01-05


In [6]:
target_col = "target_load_mw_mean"

print("Target column:", target_col)

Target column: target_load_mw_mean


In [7]:
leakage_risk_cols = [
    "load_mw_mean",
    "load_mw_max",
    "load_mw_min",
    "load_mw_std"
]

df_model = df.drop(columns=leakage_risk_cols)

print("Shape after dropping leakage-risk columns:", df_model.shape)
df_model.head()

Shape after dropping leakage-risk columns: (365, 48)


,datetime,la_temp_mean,la_temp_max,la_temp_min,la_temp_std,la_rhum_mean,la_rhum_max,la_rhum_min,la_rhum_std,la_prcp_sum,...,sd_wspd_max,sd_wspd_std,target_load_mw_mean,year,month,day,day_of_week,day_name,day_of_year,is_weekend
0,2025-01-01,11.875000,19.2,6.8,4.169272,68.125000,92,33,20.246980,0.0,...,16.6,4.159214,22574.940652,2025,1,1,2,Wednesday,1,0
1,2025-01-02,13.683333,23.6,7.7,5.340710,56.958333,84,20,22.664815,0.0,...,11.2,4.178820,24140.178717,2025,1,2,3,Thursday,2,0
2,2025-01-03,13.587500,21.4,9.4,3.157092,63.458333,81,31,12.631841,0.0,...,16.6,5.111106,24641.971448,2025,1,3,4,Friday,3,0
3,2025-01-04,12.391667,17.3,8.3,2.891504,80.875000,98,49,17.688887,0.0,...,16.6,5.137240,23242.359045,2025,1,4,5,Saturday,4,1
4,2025-01-05,13.295833,23.3,7.3,5.407039,50.875000,76,15,22.587631,0.0,...,13.0,4.360960,22584.144627,2025,1,5,6,Sunday,5,1


In [8]:
# Cyclical encoding for month
df_model["month_sin"] = np.sin(2 * np.pi * df_model["month"] / 12)
df_model["month_cos"] = np.cos(2 * np.pi * df_model["month"] / 12)

# Cyclical encoding for day_of_week
df_model["day_of_week_sin"] = np.sin(2 * np.pi * df_model["day_of_week"] / 7)
df_model["day_of_week_cos"] = np.cos(2 * np.pi * df_model["day_of_week"] / 7)

# Cyclical encoding for day_of_year
df_model["day_of_year_sin"] = np.sin(2 * np.pi * df_model["day_of_year"] / 365)
df_model["day_of_year_cos"] = np.cos(2 * np.pi * df_model["day_of_year"] / 365)

In [9]:
date_series = df_model["datetime"].copy()

In [10]:
# Drop raw calendar columns that are now redundant or less useful
cols_to_drop = [
    "datetime",   # do not feed raw timestamp directly to baseline models
    "year",       # only one year in the dataset
    "day",        # less useful once cyclical features exist
    "month",
    "day_of_week",
    "day_of_year"
]

df_model = df_model.drop(columns=cols_to_drop)

print("Shape after dropping raw calendar columns:", df_model.shape)
df_model.head()

Shape after dropping raw calendar columns: (365, 48)


,la_temp_mean,la_temp_max,la_temp_min,la_temp_std,la_rhum_mean,la_rhum_max,la_rhum_min,la_rhum_std,la_prcp_sum,la_prcp_max,...,sd_wspd_std,target_load_mw_mean,day_name,is_weekend,month_sin,month_cos,day_of_week_sin,day_of_week_cos,day_of_year_sin,day_of_year_cos
0,11.875000,19.2,6.8,4.169272,68.125000,92,33,20.246980,0.0,0.0,...,4.159214,22574.940652,Wednesday,0,0.5,0.866025,0.974928,-0.222521,0.017213,0.999852
1,13.683333,23.6,7.7,5.340710,56.958333,84,20,22.664815,0.0,0.0,...,4.178820,24140.178717,Thursday,0,0.5,0.866025,0.433884,-0.900969,0.034422,0.999407
2,13.587500,21.4,9.4,3.157092,63.458333,81,31,12.631841,0.0,0.0,...,5.111106,24641.971448,Friday,0,0.5,0.866025,-0.433884,-0.900969,0.051620,0.998667
3,12.391667,17.3,8.3,2.891504,80.875000,98,49,17.688887,0.0,0.0,...,5.137240,23242.359045,Saturday,1,0.5,0.866025,-0.974928,-0.222521,0.068802,0.997630
4,13.295833,23.3,7.3,5.407039,50.875000,76,15,22.587631,0.0,0.0,...,4.360960,22584.144627,Sunday,1,0.5,0.866025,-0.781831,0.623490,0.085965,0.996298


In [11]:
X = df_model.drop(columns=[target_col])
y = df_model[target_col]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFeature columns:")
print(X.columns.tolist())

X shape: (365, 47)
y shape: (365,)

Feature columns:
['la_temp_mean', 'la_temp_max', 'la_temp_min', 'la_temp_std', 'la_rhum_mean', 'la_rhum_max', 'la_rhum_min', 'la_rhum_std', 'la_prcp_sum', 'la_prcp_max', 'la_wspd_mean', 'la_wspd_max', 'la_wspd_std', 'sf_temp_mean', 'sf_temp_max', 'sf_temp_min', 'sf_temp_std', 'sf_rhum_mean', 'sf_rhum_max', 'sf_rhum_min', 'sf_rhum_std', 'sf_prcp_sum', 'sf_prcp_max', 'sf_wspd_mean', 'sf_wspd_max', 'sf_wspd_std', 'sd_temp_mean', 'sd_temp_max', 'sd_temp_min', 'sd_temp_std', 'sd_rhum_mean', 'sd_rhum_max', 'sd_rhum_min', 'sd_rhum_std', 'sd_prcp_sum', 'sd_prcp_max', 'sd_wspd_mean', 'sd_wspd_max', 'sd_wspd_std', 'day_name', 'is_weekend', 'month_sin', 'month_cos', 'day_of_week_sin', 'day_of_week_cos', 'day_of_year_sin', 'day_of_year_cos']


In [12]:
# Time-based split: 70% train, 15% validation, 15% test
n = len(df_model)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train = X.iloc[:train_end].copy()
y_train = y.iloc[:train_end].copy()

X_val = X.iloc[train_end:val_end].copy()
y_val = y.iloc[train_end:val_end].copy()

X_test = X.iloc[val_end:].copy()
y_test = y.iloc[val_end:].copy()

print("Train shape:", X_train.shape, y_train.shape)
print("Validation shape:", X_val.shape, y_val.shape)
print("Test shape:", X_test.shape, y_test.shape)

Train shape: (255, 47) (255,)
Validation shape: (55, 47) (55,)
Test shape: (55, 47) (55,)


In [13]:
X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)

X_val = X_val.reset_index(drop=True)
y_val = y_val.reset_index(drop=True)

X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

In [15]:
# final sanity check
print("Train date range:", date_series.iloc[:train_end].min(), "→", date_series.iloc[:train_end].max())
print("Validation date range:", date_series.iloc[train_end:val_end].min(), "→", date_series.iloc[train_end:val_end].max())
print("Test date range:", date_series.iloc[val_end:].min(), "→", date_series.iloc[val_end:].max())

print("\nMissing values:")
print("X_train:", X_train.isna().sum().sum())
print("X_val:", X_val.isna().sum().sum())
print("X_test:", X_test.isna().sum().sum())

print("\nCategorical columns:")
cat_cols = X_train.select_dtypes(include=["object", "category", "string"]).columns.tolist()
print(cat_cols)

print("\nNumeric columns:")
print(X_train.select_dtypes(include=[np.number]).columns.tolist())

Train date range: 2025-01-01 00:00:00 → 2025-09-12 00:00:00
Validation date range: 2025-09-13 00:00:00 → 2025-11-06 00:00:00
Test date range: 2025-11-07 00:00:00 → 2025-12-31 00:00:00

Missing values:
X_train: 0
X_val: 0
X_test: 0

Categorical columns:
['day_name']

Numeric columns:
['la_temp_mean', 'la_temp_max', 'la_temp_min', 'la_temp_std', 'la_rhum_mean', 'la_rhum_max', 'la_rhum_min', 'la_rhum_std', 'la_prcp_sum', 'la_prcp_max', 'la_wspd_mean', 'la_wspd_max', 'la_wspd_std', 'sf_temp_mean', 'sf_temp_max', 'sf_temp_min', 'sf_temp_std', 'sf_rhum_mean', 'sf_rhum_max', 'sf_rhum_min', 'sf_rhum_std', 'sf_prcp_sum', 'sf_prcp_max', 'sf_wspd_mean', 'sf_wspd_max', 'sf_wspd_std', 'sd_temp_mean', 'sd_temp_max', 'sd_temp_min', 'sd_temp_std', 'sd_rhum_mean', 'sd_rhum_max', 'sd_rhum_min', 'sd_rhum_std', 'sd_prcp_sum', 'sd_prcp_max', 'sd_wspd_mean', 'sd_wspd_max', 'sd_wspd_std', 'is_weekend', 'month_sin', 'month_cos', 'day_of_week_sin', 'day_of_week_cos', 'day_of_year_sin', 'day_of_year_cos']


In [16]:
import os

os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../data/artifacts", exist_ok=True)

In [17]:
# Save feature datasets
X_train.to_csv("../data/processed/X_train_daily.csv", index=False)
X_val.to_csv("../data/processed/X_val_daily.csv", index=False)
X_test.to_csv("../data/processed/X_test_daily.csv", index=False)

# Save target datasets
y_train.to_csv("../data/processed/y_train_daily.csv", index=False)
y_val.to_csv("../data/processed/y_val_daily.csv", index=False)
y_test.to_csv("../data/processed/y_test_daily.csv", index=False)

print("Unscaled datasets saved.")

Unscaled datasets saved.


## Preprocessing complete

At this stage:
- the daily dataset has been cleaned
- leakage-risk columns have been removed
- cyclical calendar features have been created
- categorical encoding and scaling have been intentionally left for the modeling pipeline
- train, validation, and test sets have been saved in chronological order

Next, I can move on to the modeling notebook and build a scikit-learn pipeline that handles:
- one-hot encoding for categorical features
- scaling for numeric features
- model training and evaluation